# Import libraries and initialize Groq

In [1]:
from groq import Groq
from dotenv import load_dotenv
import os
import json
from datetime import datetime, timezone
from pathlib import Path
import time
import pandas as pd
import numpy as np

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found")

groq_client = Groq(api_key=api_key)

MODEL_NAME = "openai/gpt-oss-20b"

print("Groq client initialized successfully")

Groq client initialized successfully


# Load Questions

In [2]:
with open("../data/questions.json", "r") as f:
    questions = json.load(f)

print(f"Total questions loaded: {len(questions)}")

for q in questions:
    print(q["id"], ":", q["question"])

Total questions loaded: 5
Q1 : What is machine learning?
Q2 : Explain the difference between supervised and unsupervised learning.
Q3 : What is overfitting in machine learning and how can it be prevented?
Q4 : Explain how a neural network works.
Q5 : What is the difference between machine learning and deep learning?


# Groq API calling Function

In [3]:
def call_llm(question, prompt_version="v1"):

    prompt = f"""
    Answer the following student question clearly.

    Question: {question}
    """

    start_time = datetime.now(timezone.utc)
    start = time.perf_counter()

    try:
        response = groq_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )

        end = time.perf_counter()
        end_time = datetime.now(timezone.utc)

        return {
            "response": response.choices[0].message.content,
            "latency": end - start,
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "question": question,
            "prompt_version": prompt_version,
            "model_name": MODEL_NAME,
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
            "response_status": "success"
        }

    except Exception as e:
        end = time.perf_counter()
        end_time = datetime.now(timezone.utc)

        return {
            "response": None,
            "latency": end - start,
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "question": question,
            "prompt_version": prompt_version,
            "model_name": MODEL_NAME,
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "response_status": f"failed: {str(e)}"
        }

# testing the API

In [4]:
result = call_llm(
    "What is machine learning?"
)

print("Response:", result["response"])
print("Latency:", round(result["latency"], 3), "seconds")
print("Input tokens:", result["input_tokens"])
print("Output tokens:", result["output_tokens"])
print("Total tokens:", result["total_tokens"])

Response: **Machine learning** is a branch of computer science that teaches computers to learn from data instead of being explicitly programmed for every task.  

- **Core idea:** The computer is given a large set of examples (data) and a goal (e.g., classify emails as spam or not spam).  
- **Learning process:** It uses mathematical algorithms to find patterns or rules in the data.  
- **Outcome:** Once trained, the model can make predictions or decisions on new, unseen data that it hasn’t seen before.

In short, machine learning lets computers improve their performance on a task by automatically discovering patterns in data.
Latency: 0.718 seconds
Input tokens: 89
Output tokens: 158
Total tokens: 247


# Execute the five academic questions

In [5]:
results = []

for item in questions:

    print(f"Processing {item['id']}...")

    result = call_llm(
        item["question"],
        prompt_version="v1"
    )

    result["question_id"] = item["id"]

    results.append(result)

    print("Status:", result["response_status"])
    print("Latency:", round(result["latency"], 3), "seconds")
    print("-" * 50)

Processing Q1...
Status: success
Latency: 0.839 seconds
--------------------------------------------------
Processing Q2...
Status: success
Latency: 0.939 seconds
--------------------------------------------------
Processing Q3...
Status: success
Latency: 1.545 seconds
--------------------------------------------------
Processing Q4...
Status: success
Latency: 1.779 seconds
--------------------------------------------------
Processing Q5...
Status: success
Latency: 1.135 seconds
--------------------------------------------------


# Latency statistics

In [6]:
df = pd.DataFrame(results)

latency_df = df[
    [
        "question_id",
        "prompt_version",
        "model_name",
        "start_time",
        "end_time",
        "latency",
        "response_status"
    ]
]

display(latency_df)

,question_id,prompt_version,model_name,start_time,end_time,latency,response_status
0,Q1,v1,openai/gpt-oss-20b,2026-10-31T10:05:25.842855+00:00,2026-10-31T10:05:26.681452+00:00,0.838587,success
1,Q2,v1,openai/gpt-oss-20b,2026-10-31T10:05:26.682140+00:00,2026-10-31T10:05:27.620728+00:00,0.938581,success
2,Q3,v1,openai/gpt-oss-20b,2026-10-31T10:05:27.621491+00:00,2026-10-31T10:05:29.166751+00:00,1.545251,success
3,Q4,v1,openai/gpt-oss-20b,2026-10-31T10:05:29.167136+00:00,2026-10-31T10:05:30.945645+00:00,1.778501,success
4,Q5,v1,openai/gpt-oss-20b,2026-10-31T10:05:30.946004+00:00,2026-10-31T10:05:32.081287+00:00,1.135277,success


In [7]:
latency_stats = {
    "Mean": df["latency"].mean(),
    "Median": df["latency"].median(),
    "Minimum": df["latency"].min(),
    "Maximum": df["latency"].max(),
    "Standard Deviation": df["latency"].std()
}

latency_summary = pd.DataFrame(
    latency_stats.items(),
    columns=["Metric", "Latency (seconds)"]
)

display(latency_summary.round(4))

,Metric,Latency (seconds)
0,Mean,1.2472
1,Median,1.1353
2,Minimum,0.8386
3,Maximum,1.7785
4,Standard Deviation,0.4018


In [8]:
slowest = latency_df.loc[
    latency_df["latency"].idxmax()
]

print("Slowest Question:", slowest["question_id"])
print("Latency:", round(slowest["latency"], 4), "seconds")

Slowest Question: Q4
Latency: 1.7785 seconds


### Latency Interpretation

The five API requests achieved an average response latency of
approximately 1.24 seconds, with a median of 1.13 seconds.
Q4, which asked about how a neural network works, was the
slowest request, taking approximately 1.778 seconds. The higher latency may be
attributed to the longer generated response, model processing
time, network conditions, or temporary service load. Overall,
the observed latency indicates that the selected Groq model
can respond to academic queries within a relatively short
time under the tested conditions.

# Price Definition

In [9]:
INPUT_PRICE_PER_MILLION = 0.075
OUTPUT_PRICE_PER_MILLION = 0.30

**Pricing assumptions**

- Provider: Groq API
- Model: openai/gpt-oss-20b
- Currency: USD
- Input token price: $0.075 per million tokens
- Output token price: $0.30 per million tokens
- Pricing reference date: 31-10-2026
- Note: Actual costs may vary if provider pricing changes.

## Request Level Cost

The formulas are:

**Input Cost:**

$$
InputCost = \frac{InputTokens}{1,000,000}
\times InputPrice
$$

**Output Cost:**

$$
OutputCost = \frac{OutputTokens}{1,000,000}
\times OutputPrice
$$

**Total Cost per Request:**

$$
TotalCost = InputCost + OutputCost
$$



In [10]:
df["input_cost_usd"] = (
    df["input_tokens"] / 1_000_000
) * INPUT_PRICE_PER_MILLION

df["output_cost_usd"] = (
    df["output_tokens"] / 1_000_000
) * OUTPUT_PRICE_PER_MILLION

df["total_cost_usd"] = (
    df["input_cost_usd"] + df["output_cost_usd"]
)

## Projected Monthly Cost

In [11]:
successful_requests = df[
    df["response_status"] == "success"
]

average_cost = successful_requests[
    "total_cost_usd"
].mean()

monthly_requests = 10_000

monthly_projected_cost = (
    average_cost * monthly_requests
)

print("Average cost per request: $", average_cost)
print("Projected monthly cost for 10,000 requests: $", monthly_projected_cost)

Average cost per request: $ 0.000224295
Projected monthly cost for 10,000 requests: $ 2.24295


## Cost Calculations / Query

In [12]:
cost_df = df[
    [
        "question_id",
        "input_tokens",
        "output_tokens",
        "total_tokens",
        "input_cost_usd",
        "output_cost_usd",
        "total_cost_usd"
    ]
]

display(cost_df.round(8))

,question_id,input_tokens,output_tokens,total_tokens,input_cost_usd,output_cost_usd,total_cost_usd
0,Q1,89,333,422,0.000007,0.000100,0.000107
1,Q2,95,581,676,0.000007,0.000174,0.000181
2,Q3,99,976,1075,0.000007,0.000293,0.000300
3,Q4,91,1195,1286,0.000007,0.000358,0.000365
4,Q5,95,536,631,0.000007,0.000161,0.000168


# Generate CSV

In [13]:
os.makedirs("../outputs", exist_ok=True)

df.to_csv(
    "../outputs/latency_tokens_cost.csv",
    index=False
)

print("CSV exported successfully!")

CSV exported successfully!


In [14]:
latency_summary.to_csv(
    "../outputs/latency_summary.csv",
    index=False
)

print("CSV exported successfully!")

CSV exported successfully!


# Performance & Cost Metrics

* **Average Response Latency:** ~1.247 seconds
* **Average API Cost:** ~\$0.000224295 per request
* **Projected Monthly Expenditure:** ~\$2.24295 (based on 10,000 requests/month)

#### Key Observations
* **Slowest Request (Q4):** Took **1.7785 seconds** with **1195 output tokens**. 
* **Latency Factors:** The higher response time for Q4 is likely due to the longer generated response, alongside standard variables like network conditions and model processing time.

*Note: Projections assume consistent token consumption and unchanged provider pricing.*


## Cost Optimization Recommendation

To reduce API costs, use a smaller LLM for simple academic queries and reserve larger models for complex questions. Additionally, limit unnecessary prompt instructions and response length to reduce token consumption. Caching frequently used prompt content and responses can further minimize repeated token usage and improve overall cost efficiency.

# MLflow Prompt Versioning and Evaluation

In [15]:
import mlflow
from mlflow import MlflowClient

print("MLflow version:", mlflow.__version__)

MLflow version: 3.17.2


## Configure MLflow tracking

In [16]:
# Configure MLflow tracking

mlflow.set_tracking_uri("sqlite:///../mlflow.db")

EXPERIMENT_NAME = "eduassist_llmops_assignment"

mlflow.set_experiment(EXPERIMENT_NAME)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Active Experiment:", EXPERIMENT_NAME)

Tracking URI: sqlite:///../mlflow.db
Active Experiment: eduassist_llmops_assignment


## Define prompt Versions

In [17]:
PROMPT_NAME = "eduassist_qa_prompt"

prompt_templates = {

    "v1": """
Answer the following student question clearly.

Question: {{question}}
""",

    "v2": """
You are an academic assistant.

Answer the following student question in a clear,
concise, and accurate manner.

Use simple language suitable for undergraduate students.

Question: {{question}}
""",

    "v3": """
You are an academic assistant.

Answer the student question and return ONLY valid JSON.

Do not include markdown, explanations outside JSON,
or code fences.

Use this exact JSON schema:

{
    "answer": "string",
    "confidence": "high | medium | low",
    "key_points": ["string"]
}

Question: {{question}}
"""
}

In [18]:
os.makedirs("../prompts", exist_ok=True)

for version, template in prompt_templates.items():

    file_path = f"../prompts/prompt_{version}.txt"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(template.strip())

print("All three prompt templates saved successfully!")

All three prompt templates saved successfully!


## Register prompts in Mlflow

In [19]:
# Register each prompt only if an identical template is not already present.
# This makes the cell safe to rerun without creating duplicate prompt versions.

commit_messages = {
    "v1": "Initial basic academic question answering prompt",
    "v2": "Improved prompt with academic assistant persona and structured instructions",
    "v3": "Added strict JSON output instructions and response schema"
}

client = MlflowClient()

# Load the latest registered version, if this prompt already exists.
latest_prompt = mlflow.genai.load_prompt(
    PROMPT_NAME,
    allow_missing=True
)

existing_versions_by_template = {}

if latest_prompt is not None:
    latest_version_number = int(latest_prompt.version)

    # Read the existing version history so older matching templates can be reused.
    for version_number in range(1, latest_version_number + 1):
        existing_prompt = mlflow.genai.load_prompt(
            f"prompts:/{PROMPT_NAME}/{version_number}",
            allow_missing=True
        )

        if existing_prompt is not None:
            existing_versions_by_template[existing_prompt.template.strip()] = int(
                existing_prompt.version
            )

prompt_versions = {}

for version, template in prompt_templates.items():
    normalized_template = template.strip()

    if normalized_template in existing_versions_by_template:
        # Reuse the existing version instead of registering a duplicate.
        registry_version = existing_versions_by_template[normalized_template]
        print(
            f"{version} already exists | "
            f"Reusing MLflow Version {registry_version}"
        )
    else:
        # Register only when this exact template is not already in the registry.
        registered_prompt = mlflow.genai.register_prompt(
            name=PROMPT_NAME,
            template=normalized_template,
            commit_message=commit_messages[version],
            tags={
                "assignment": "LLMOps Assignment 2",
                "prompt_variant": version,
                "model": MODEL_NAME
            }
        )

        registry_version = int(registered_prompt.version)

        # Add the new version to the lookup so repeated variants in this cell
        # cannot be registered twice.
        existing_versions_by_template[normalized_template] = registry_version

        print(
            f"{version} registered successfully | "
            f"Name: {registered_prompt.name} | "
            f"MLflow Version: {registry_version}"
        )

    prompt_versions[version] = registry_version

print("\nPrompt version mapping:", prompt_versions)


v1 already exists | Reusing MLflow Version 1
v2 already exists | Reusing MLflow Version 2
v3 already exists | Reusing MLflow Version 3

Prompt version mapping: {'v1': 1, 'v2': 2, 'v3': 3}


In [20]:
for label, version in prompt_versions.items():

    prompt = mlflow.genai.load_prompt(
        f"prompts:/{PROMPT_NAME}/{version}"
    )

    print(f"\n{label} - MLflow Version {version}")
    print("-" * 50)
    print(prompt.template)


v1 - MLflow Version 1
--------------------------------------------------
Answer the following student question clearly.

Question: {{question}}

v2 - MLflow Version 2
--------------------------------------------------
You are an academic assistant.

Answer the following student question in a clear,
concise, and accurate manner.

Use simple language suitable for undergraduate students.

Question: {{question}}

v3 - MLflow Version 3
--------------------------------------------------
You are an academic assistant.

Answer the student question and return ONLY valid JSON.

Do not include markdown, explanations outside JSON,
or code fences.

Use this exact JSON schema:

{
    "answer": "string",
    "confidence": "high | medium | low",
    "key_points": ["string"]
}

Question: {{question}}


## Validate Response Schema

In [21]:
from mlflow.genai import scorer

In [22]:
def is_valid_eduassist_json(response_text):

    try:
        data = json.loads(response_text)

        # Check that the response is a JSON object
        if not isinstance(data, dict):
            return False

        # Check required fields
        required_fields = {
            "answer",
            "confidence",
            "key_points"
        }

        if set(data.keys()) != required_fields:
            return False

        # Validate field types and values
        if not isinstance(data["answer"], str):
            return False

        if data["confidence"] not in [
            "high", "medium", "low"
        ]:
            return False

        if not isinstance(data["key_points"], list):
            return False

        if not all(
            isinstance(point, str)
            for point in data["key_points"]
        ):
            return False

        return True

    except (json.JSONDecodeError, TypeError):
        return False

## Create a function to call a registered prompt

In [23]:
def call_registered_prompt(
    question,
    question_id,
    prompt_label,
    registry_version,
    prompt
):

    # Populate the registered prompt template
    formatted_prompt = prompt.format(
        question=question
    )

    start_time = datetime.now(timezone.utc)
    start = time.perf_counter()

    try:

        response = groq_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "user",
                    "content": formatted_prompt
                }
            ],
            temperature=0.2
        )

        end = time.perf_counter()
        end_time = datetime.now(timezone.utc)

        response_text = response.choices[0].message.content

        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens
        total_tokens = response.usage.total_tokens

        input_cost = (
            input_tokens / 1_000_000
        ) * INPUT_PRICE_PER_MILLION

        output_cost = (
            output_tokens / 1_000_000
        ) * OUTPUT_PRICE_PER_MILLION

        total_cost = input_cost + output_cost

        return {
            "question_id": question_id,
            "question": question,
            "prompt_version": prompt_label,
            "registry_version": registry_version,
            "model_name": MODEL_NAME,
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "latency": end - start,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "input_cost_usd": input_cost,
            "output_cost_usd": output_cost,
            "total_cost_usd": total_cost,
            "response": response_text,
            "valid_json": is_valid_eduassist_json(response_text),
            "response_status": "success"
        }

    except Exception as e:

        end = time.perf_counter()
        end_time = datetime.now(timezone.utc)

        return {
            "question_id": question_id,
            "question": question,
            "prompt_version": prompt_label,
            "registry_version": registry_version,
            "model_name": MODEL_NAME,
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "latency": end - start,
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "input_cost_usd": 0,
            "output_cost_usd": 0,
            "total_cost_usd": 0,
            "response": None,
            "valid_json": False,
            "response_status": f"failed: {str(e)}"
        }

## Execute all three prompt versions

In [24]:
all_eval_results = []

run_ids = {}

for label, registry_version in prompt_versions.items():

    print(f"\nEvaluating {label}")
    print("=" * 50)

    # Load the specific registered prompt version
    prompt = mlflow.genai.load_prompt(
        f"prompts:/{PROMPT_NAME}/{registry_version}"
    )

    version_results = []

    # Create one MLflow run per prompt version
    with mlflow.start_run(
        run_name=f"prompt_{label}_evaluation"
    ) as run:

        run_ids[label] = run.info.run_id

        mlflow.log_params({
            "prompt_name": PROMPT_NAME,
            "prompt_version": label,
            "registry_version": registry_version,
            "model_name": MODEL_NAME,
            "provider": "Groq",
            "temperature": 0.2,
            "number_of_questions": len(questions)
        })

        for item in questions:

            print(f"Processing {item['id']}...")

            result = call_registered_prompt(
                question=item["question"],
                question_id=item["id"],
                prompt_label=label,
                registry_version=registry_version,
                prompt=prompt
            )

            version_results.append(result)
            all_eval_results.append(result)

            print("Status:", result["response_status"])
            print("Latency:", round(result["latency"], 3))
            print("Tokens:", result["total_tokens"])
            print("Valid JSON:", result["valid_json"])

        # Calculate metrics using successful API responses
        version_df = pd.DataFrame(version_results)

        successful_df = version_df[
            version_df["response_status"] == "success"
        ]

        if successful_df.empty:
            raise RuntimeError(
                f"All API calls failed for {label}. "
                "Check the API and rerun this version."
            )

        metrics = {
            "avg_latency_seconds": successful_df["latency"].mean(),
            "avg_input_tokens": successful_df["input_tokens"].mean(),
            "avg_output_tokens": successful_df["output_tokens"].mean(),
            "avg_cost_per_request": successful_df["total_cost_usd"].mean(),
            "valid_json": successful_df["valid_json"].mean()
        }

        # Log required metrics in MLflow
        mlflow.log_metrics(metrics)

        # Log additional operational information
        mlflow.log_metrics({
            "successful_requests": len(successful_df),
            "total_requests": len(version_df)
        })

        # Save prompt text as an MLflow run artifact
        mlflow.log_text(
            prompt.template,
            f"prompts/{label}.txt"
        )

        # Save individual responses for traceability
        mlflow.log_text(
            json.dumps(
                version_results,
                indent=4
            ),
            f"responses/{label}_responses.json"
        )

        print("\nMetrics for", label)

        for metric, value in metrics.items():
            print(f"{metric}: {value:.4f}")

print("\nAll prompt evaluations completed!")


Evaluating v1
Processing Q1...
Status: success
Latency: 0.788
Tokens: 362
Valid JSON: False
Processing Q2...
Status: success
Latency: 1.364
Tokens: 938
Valid JSON: False
Processing Q3...
Status: success
Latency: 1.126
Tokens: 758
Valid JSON: False
Processing Q4...
Status: success
Latency: 1.763
Tokens: 1355
Valid JSON: False
Processing Q5...
Status: success
Latency: 0.907
Tokens: 555
Valid JSON: False

Metrics for v1
avg_latency_seconds: 1.1898
avg_input_tokens: 89.8000
avg_output_tokens: 703.8000
avg_cost_per_request: 0.0002
valid_json: 0.0000

Evaluating v2
Processing Q1...
Status: success
Latency: 0.743
Tokens: 384
Valid JSON: False
Processing Q2...
Status: success
Latency: 0.962
Tokens: 457
Valid JSON: False
Processing Q3...
Status: success
Latency: 1.032
Tokens: 716
Valid JSON: False
Processing Q4...
Status: success
Latency: 6.553
Tokens: 995
Valid JSON: False
Processing Q5...
Status: success
Latency: 7.112
Tokens: 540
Valid JSON: False

Metrics for v2
avg_latency_seconds: 3.2802

In [25]:
prompt_results_df = pd.DataFrame(all_eval_results)

display(
    prompt_results_df[
        [
            "question_id",
            "prompt_version",
            "latency",
            "input_tokens",
            "output_tokens",
            "total_cost_usd",
            "valid_json",
            "response_status"
        ]
    ]
)

,question_id,prompt_version,latency,input_tokens,output_tokens,total_cost_usd,valid_json,response_status
0,Q1,v1,0.788299,85,277,0.000089,False,success
1,Q2,v1,1.363782,91,847,0.000261,False,success
2,Q3,v1,1.126296,95,663,0.000206,False,success
3,Q4,v1,1.763204,87,1268,0.000387,False,success
4,Q5,v1,0.907293,91,464,0.000146,False,success
5,Q1,v2,0.742503,108,276,0.000091,False,success
6,Q2,v2,0.961973,114,343,0.000111,False,success
7,Q3,v2,1.031555,118,598,0.000188,False,success
8,Q4,v2,6.553327,110,885,0.000274,False,success
9,Q5,v2,7.111541,114,426,0.000136,False,success


## Export the prompt results

In [26]:
os.makedirs("../outputs", exist_ok=True)

prompt_results_df.to_csv(
    "../outputs/prompt_evaluation_results.csv",
    index=False
)

print("Prompt evaluation results exported!")

Prompt evaluation results exported!


## MLflow's GenAI evaluation scorer

In [27]:
@scorer
def valid_json(outputs: str) -> bool:

    return is_valid_eduassist_json(outputs)

In [28]:
genai_eval_results = {}

for label in prompt_versions:

    version_df = prompt_results_df[
        (prompt_results_df["prompt_version"] == label) &
        (prompt_results_df["response_status"] == "success")
    ]

    eval_data = [
        {
            "inputs": {
                "question": row["question"]
            },
            "outputs": row["response"]
        }
        for _, row in version_df.iterrows()
    ]

    evaluation = mlflow.genai.evaluate(
        data=eval_data,
        scorers=[valid_json]
    )

    genai_eval_results[label] = evaluation

    print(f"\nMLflow GenAI evaluation completed for {label}")
    print(evaluation)

2026/10/31 15:36:15 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.



✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: learned-skunk-609
  Run ID: 6217def406294ce7bbd9368f5b5a99c0

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.


MLflow GenAI evaluation completed for v1
EvaluationResult(
  run_id: 6217def406294ce7bbd9368f5b5a99c0
  metrics:
    valid_json/mean: 0.0
  result_df: 5 rows x 13 cols
)

✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: youthful-colt-76
  Run ID: 83117eee2ad04b8ba7b140a3a951a068

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.


MLflow GenAI evaluation completed for v2
EvaluationResult(
  run_id: 83117eee2ad04b8ba7b140a3a951a068
  metrics:
    valid_json/mean: 0.0
  result_df: 5 rows x 13 cols
)

✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name

## Prompt comparison table

In [29]:
comparison_rows = []

for label in prompt_versions:

    version_df = prompt_results_df[
        (prompt_results_df["prompt_version"] == label) &
        (prompt_results_df["response_status"] == "success")
    ]

    comparison_rows.append({

        "Prompt Version": label,

        "Avg Latency": version_df["latency"].mean(),

        "Avg Input Tokens": version_df["input_tokens"].mean(),

        "Avg Output Tokens": version_df["output_tokens"].mean(),

        "Avg Cost": version_df["total_cost_usd"].mean(),

        "Valid JSON Rate": version_df["valid_json"].mean()

    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df.round(5))

,Prompt Version,Avg Latency,Avg Input Tokens,Avg Output Tokens,Avg Cost,Valid JSON Rate
0,v1,1.18977,89.8,703.8,0.00022,0.0
1,v2,3.28018,112.8,505.6,0.00016,0.0
2,v3,3.27128,145.8,294.8,0.00010,1.0


In [30]:
comparison_df.to_csv(
    "../outputs/prompt_comparison.csv",
    index=False
)

print("Evaluation results exported!")

Evaluation results exported!


## Inspect the Prompt Outputs

In [31]:
for label in prompt_versions:

    print("\n", "=" * 70)
    print("PROMPT VERSION:", label)
    print("=" * 70)

    version_df = prompt_results_df[
        prompt_results_df["prompt_version"] == label
    ]

    for _, row in version_df.iterrows():

        print("\nQuestion:", row["question_id"])
        print("Response:", row["response"])
        print("Valid JSON:", row["valid_json"])
        print("-" * 50)


PROMPT VERSION: v1

Question: Q1
Response: **Machine learning** is a field of computer science that teaches computers to learn patterns and make decisions from data, rather than being explicitly programmed for every task.

Key points:

| Aspect | What it means |
|--------|---------------|
| **Learning from data** | The system is fed examples (images, text, numbers, etc.) and it discovers regularities. |
| **Improving over time** | As it sees more data, its predictions or actions become more accurate. |
| **Algorithms** | Common types include: <br>• **Supervised learning** – learns a mapping from inputs to known outputs (e.g., spam detection). <br>• **Unsupervised learning** – finds hidden structure in unlabeled data (e.g., clustering customers). <br>• **Reinforcement learning** – learns by trial‑and‑error to maximize a reward (e.g., game playing). |
| **Applications** | Image and speech recognition, recommendation engines, autonomous vehicles, medical diagnosis, financial forecasting,

## Print Mlflow Configurations

In [32]:
from pathlib import Path
import mlflow
from mlflow import MlflowClient

db_path = Path("../mlflow.db").resolve()

print("Notebook working directory:", Path.cwd())
print("Expected MLflow database:", db_path)
print("Database exists:", db_path.exists())
print("Tracking URI:", mlflow.get_tracking_uri())

client = MlflowClient()
experiment = client.get_experiment_by_name(
    "eduassist_llmops_assignment"
)

print("Experiment found:", experiment is not None)

if experiment:
    print("Experiment ID:", experiment.experiment_id)

Notebook working directory: C:\dwai_github\LLMops\notebooks
Expected MLflow database: C:\dwai_github\LLMops\mlflow.db
Database exists: True
Tracking URI: sqlite:///../mlflow.db
Experiment found: True
Experiment ID: 1


# Production promotion

In [33]:
BEST_VERSION = 3

client.set_prompt_alias(
    name=PROMPT_NAME,
    alias="production",
    version=BEST_VERSION
)

print(f"Production alias assigned to version {BEST_VERSION}")

Production alias assigned to version 3


In [34]:
production_prompt = mlflow.genai.load_prompt(
    f"prompts:/{PROMPT_NAME}@production"
)

print("Production prompt loaded successfully!")
print(production_prompt)

Production prompt loaded successfully!
PromptVersion(name=eduassist_qa_prompt, version=3, template=You are an academic assistant....)


### Production Promotion Justification – Version 3

Version 3 was promoted to production because it demonstrated the best overall performance among the three evaluated prompt versions. It achieved **100% valid JSON responses (5/5)**, ensuring consistent and structured outputs suitable for downstream application integration.

It also recorded lowest average output tokens (294.8), and lowest average cost per request ($0.00010). Although it required slightly more input tokens due to additional JSON formatting instructions, the reduction in output tokens resulted in better overall cost efficiency.

Considering its superior JSON compliance, faster response time, reduced output token consumption, and lower API cost, Version 3 was selected as the most suitable prompt for production deployment.